# 05 — Drift Monitoring
## NovaSend Fulfillment Risk Pipeline

**Purpose**: Detect distributional shift between training data and incoming production orders.
If monitored features drift significantly, model predictions become unreliable and retraining should be triggered.

**Method**: Population Stability Index (PSI)
**Monitored features**:
- `Days_for_shipment_scheduled` — strongest numeric predictor; sensitive to carrier or ops changes
- `Order_Status_SUSPECTED_FRAUD` — fraud patterns are known to shift over time
- `Shipping_Mode_Same_Day` — shipping behavior can shift seasonally
- `discount_tier_aggressive` — sensitive to promotional strategy changes

**PSI Thresholds**:
| PSI Value | Interpretation | Action |
|---|---|---|
| < 0.10 | No significant shift | No action |
| 0.10 — 0.20 | Moderate shift | Monitor closely |
| > 0.20 | Significant shift | Investigate and consider retraining |

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")  # suppress non-critical warnings for clean notebook output

# Spark is pre-initialized in Databricks as 'spark' — no SparkSession.builder needed
print("Libraries loaded successfully.")

In [0]:
#Load Training Baseline
# Loading the engineered feature table from Delta as my training baseline reference
df_train = spark.table("novasend_fulfillment.processed.features_engineered").toPandas()

print(f"Training baseline loaded: {df_train.shape[0]:,} rows, {df_train.shape[1]} columns")
print(df_train[["Days_for_shipment_scheduled", "Order_Status_SUSPECTED_FRAUD", 
                 "Shipping_Mode_Same_Day", "discount_tier_aggressive"]].describe())

In [0]:
#PSI Calculation Function
def calculate_psi(baseline, production, bins=10, epsilon=1e-4):
    # Continuous feature PSI using percentile-based binning
    breakpoints = np.linspace(0, 100, bins + 1)
    bin_edges = np.percentile(baseline, breakpoints)
    bin_edges = np.unique(bin_edges)

    baseline_counts, _ = np.histogram(baseline, bins=bin_edges)
    production_counts, _ = np.histogram(production, bins=bin_edges)

    baseline_pct = baseline_counts / len(baseline) + epsilon
    production_pct = production_counts / len(production) + epsilon

    psi_values = (production_pct - baseline_pct) * np.log(production_pct / baseline_pct)
    return psi_values.sum()


def calculate_psi_binary(baseline, production, epsilon=1e-4):
    # Binary feature PSI — compares proportion of 1s directly instead of using histograms
    # Histogram binning fails for binary columns with low prevalence
    p_base = baseline.mean() + epsilon
    p_prod = production.mean() + epsilon

    # Also evaluate the 0-class proportion to capture both sides of the distribution
    q_base = (1 - baseline.mean()) + epsilon
    q_prod = (1 - production.mean()) + epsilon

    psi = (p_prod - p_base) * np.log(p_prod / p_base) + \
          (q_prod - q_base) * np.log(q_prod / q_base)
    return round(psi, 4)

print("PSI functions defined.")

In [0]:
# Simulate Production Sample

np.random.seed(42)  # seed set for reproducibility
n_production = 5000  # realistic daily order volume for a single fulfillment center

df_prod = pd.DataFrame()

# Days_for_shipment_scheduled — shifted upward to simulate a carrier capacity crunch
# Training mean was ~2.93, production mean pushed to ~3.6
df_prod["Days_for_shipment_scheduled"] = np.clip(
    np.random.normal(loc=3.6, scale=1.2, size=n_production).round().astype(int),
    0, 6
)

# Order_Status_SUSPECTED_FRAUD — rate doubled to simulate a new fraud wave
# Training rate was ~11%, production pushed to ~22%
df_prod["Order_Status_SUSPECTED_FRAUD"] = np.random.binomial(
    n=1, p=0.22, size=n_production
).astype(int)

# Shipping_Mode_Same_Day — stable, matches training rate of ~8%
df_prod["Shipping_Mode_Same_Day"] = np.random.binomial(
    n=1, p=0.08, size=n_production
).astype(int)

# discount_tier_aggressive — stable, matches training rate of ~11%
df_prod["discount_tier_aggressive"] = np.random.binomial(
    n=1, p=0.11, size=n_production
).astype(int)

print(f"Production sample created: {df_prod.shape[0]:,} rows")
print(df_prod.describe())

In [0]:
# Calculate PSI for All Monitored Features

# Binary features use proportion comparison — histogram binning fails at low prevalence
binary_features = [
    "Order_Status_SUSPECTED_FRAUD",
    "Shipping_Mode_Same_Day",
    "discount_tier_aggressive"
]

# Continuous features use percentile-based histogram binning
continuous_features = [
    "Days_for_shipment_scheduled"
]

psi_results = {}

for feature in continuous_features:
    psi_results[feature] = calculate_psi(
        baseline=df_train[feature].values,
        production=df_prod[feature].values
    )

for feature in binary_features:
    psi_results[feature] = calculate_psi_binary(
        baseline=df_train[feature].values,
        production=df_prod[feature].values
    )

def get_psi_status(psi):
    if psi < 0.10:
        return "Stable"
    elif psi < 0.20:
        return "Moderate Shift"
    else:
        return "Significant Shift"

df_psi = pd.DataFrame([
    {"Feature": f, "PSI": round(psi_results[f], 4), "Status": get_psi_status(psi_results[f])}
    for f in continuous_features + binary_features
])

print(df_psi.to_string(index=False))

In [0]:
print(df_train["Order_Status_SUSPECTED_FRAUD"].value_counts())
print(df_train["Order_Status_SUSPECTED_FRAUD"].mean())

In [0]:
# PSI Results Visualization

fig, ax = plt.subplots(figsize=(10, 5))

# Color each bar based on drift status
colors = [
    "#d9534f" if psi > 0.20 else "#f0ad4e" if psi > 0.10 else "#5cb85c"
    for psi in df_psi["PSI"]
]

bars = ax.barh(df_psi["Feature"], df_psi["PSI"], color=colors)

# Threshold lines so the viewer can read status without referencing the table
ax.axvline(x=0.10, color="#f0ad4e", linestyle="--", linewidth=1.2, label="Moderate Shift (0.10)")
ax.axvline(x=0.20, color="#d9534f", linestyle="--", linewidth=1.2, label="Significant Shift (0.20)")

# PSI score labels on each bar
for bar, psi in zip(bars, df_psi["PSI"]):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{psi:.4f}",
        va="center",
        fontsize=10
    )

ax.set_xlabel("PSI Score", fontsize=11)
ax.set_title("Feature Drift Detection — NovaSend Fulfillment Risk Model", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim(0, max(df_psi["PSI"]) + 0.15)

plt.tight_layout()
plt.show()

# Drift Analysis Conclusion

## Results Summary

| Feature | PSI | Status |
|---|---|---|
| Days_for_shipment_scheduled | 0.2849 | Significant Shift |
| Order_Status_SUSPECTED_FRAUD | 0.4637 | Significant Shift |
| Shipping_Mode_Same_Day | 0.0141 | Stable |
| discount_tier_aggressive | 0.0003 | Stable |

## Interpretation

Two features flagged significant distributional shift in the simulated production sample.

`Days_for_shipment_scheduled` shifted from a training mean of 2.93 days to a production mean of 3.60 days. This pattern is consistent with a carrier capacity crunch compressing fulfillment windows across the network.

`Order_Status_SUSPECTED_FRAUD` shifted from a training prevalence of 2.25% to a simulated production rate of 22%. This magnitude of shift suggests a new fraud pattern has emerged that was not well represented in training data.

## Recommended Action

Both flagged features are significant inputs to the model. Distributional shift of this magnitude is likely degrading prediction reliability. The recommended response is to collect a labeled sample of recent production orders, retrain the model on combined historical and recent data, and re-evaluate AUC on a held-out slice of recent orders before promoting the new model to production.

## Features Confirmed Stable

`Shipping_Mode_Same_Day` and `discount_tier_aggressive` show no meaningful shift. No action required on these features.